In [5]:
import pandas as pd

In [ ]:
df_times2 = pd.read_csv("../workspace/results_V11/REMIND-MAgPIE_DeepElec__asset_granularity_with_staggered_shock_and_retirement/company_technology_npv.csv") 
df_ratio = pd.read_csv("../data/07_model_output/company_technology_npv_theta1.csv") 


In [15]:
companies_comparison = pd.merge(
    df_times2[
    ["company_id", "company_name", "sector", 
    "technology", "scenario_geography", "npv_change" ]].rename(columns={"npv_change": "times2_npv_change"}),
    df_ratio[
    ["company_id", "company_name", "sector", 
    "technology", "scenario_geography", "npv_change" ]].rename(columns={"npv_change": "ratio_npv_change"}),
    how="inner",
)
companies_comparison["npv_change_diff"] = companies_comparison["ratio_npv_change"] - companies_comparison["times2_npv_change"] 
companies_comparison

,company_id,company_name,sector,technology,scenario_geography,times2_npv_change,ratio_npv_change,npv_change_diff
0,CN_1010711778297954678,Tanzania Electric Supply Company LTD (TANESCO),Power,HydroCap,R10AFRICA,-0.216685,-0.824494,-0.607809
1,CN_1010711778297954678,Tanzania Electric Supply Company LTD (TANESCO),Power,SolarCap - PV,R10AFRICA,-0.700812,-0.788252,-0.087440
2,CN_1165136230475758831,CEE Group,Power,SolarCap - PV,EU,-0.682099,-0.784243,-0.102144
3,CN_1165136230475758831,CEE Group,Power,SolarCap - PV,R10EUROPE,-0.682099,-0.784243,-0.102144
4,CN_1246771537420033421,UAB Ignitis Renewables,Power,SolarCap - PV,EU,-0.602949,-0.607682,-0.004733
...,...,...,...,...,...,...,...,...
512,CP_908081418883864072,Octopus Renewables Ltd,Power,BiomassCap - w/o CCS,R10EUROPE,-0.139502,-0.079561,0.059941
513,CP_9121337298268767292,Rosatom,Power,NuclearCap,R10REF_ECON,-0.644636,-0.727788,-0.083153
514,CP_9121337298268767292,Rosatom,Power,NuclearCap,RUS,-0.071149,-0.263535,-0.192386
515,CP_977676203267714574,ERG SpA,Power,GasCap - w/o CCS,EU,-0.562468,-0.632924,-0.070456


In [16]:
result = companies_comparison.groupby(["scenario_geography", "technology"])["npv_change_diff"].agg([
    ('mean', 'mean'),
    ('median', 'median'),
    ('min', 'min'),
    ('max', 'max'),
    ('percentile_2.5', lambda x: x.quantile(0.025)),
    ('percentile_5', lambda x: x.quantile(0.05)),
    ('percentile_95', lambda x: x.quantile(0.95)),
    ('percentile_97.5', lambda x: x.quantile(0.975))
]).reset_index()
result = result.sort_values(by=["technology", "scenario_geography"], ascending=False)
result

,scenario_geography,technology,mean,median,min,max,percentile_2.5,percentile_5,percentile_95,percentile_97.5
91,USA,SolarCap - PV,0.130714,0.129757,0.129757,0.150614,0.129757,0.129757,0.133849,0.140551
83,RUS,SolarCap - PV,-0.772680,-0.772680,-0.772680,-0.772680,-0.772680,-0.772680,-0.772680,-0.772680
80,R10REST_ASIA,SolarCap - PV,-0.100414,-0.105002,-0.105002,-0.045347,-0.105002,-0.105002,-0.081140,-0.063244
76,R10REF_ECON,SolarCap - PV,-0.711868,-0.783431,-0.783431,-0.210932,-0.783431,-0.783431,-0.411307,-0.311119
72,R10PAC_OECD,SolarCap - PV,-0.363859,-0.365104,-0.365104,-0.340214,-0.365104,-0.365104,-0.363859,-0.352037
...,...,...,...,...,...,...,...,...,...,...
34,R10CHINA+,BiomassCap - w/o CCS,0.205993,0.205993,0.205993,0.205993,0.205993,0.205993,0.205993,0.205993
26,R10AFRICA,BiomassCap - w/o CCS,-0.584630,-0.584630,-0.584630,-0.584630,-0.584630,-0.584630,-0.584630,-0.584630
19,JPN,BiomassCap - w/o CCS,-0.038548,-0.038548,-0.038548,-0.038548,-0.038548,-0.038548,-0.038548,-0.038548
4,EU,BiomassCap - w/o CCS,0.069821,0.066603,0.059941,0.086138,0.059941,0.059941,0.084207,0.085173


In [17]:
result.to_csv("../comparison_pricetimes2_vs_ratio.csv", index=False)